In [2]:
import os
if "COLAB_" not in "".join(os.environ.keys()):
    # Kaggle path
    !pip install -q --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
    !pip install -q --no-deps cut_cross_entropy unsloth_zoo
    !pip install -q sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install -q --no-deps unsloth
    !pip install -q --upgrade transformers
print("Unsloth stack installed.")

Unsloth stack installed.


In [4]:
!pip install -q unsloth
print("---")
import unsloth
print(f"Unsloth: {unsloth.__version__}")

---
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: 2026.7.3


In [5]:
import subprocess
result = subprocess.run(["pip", "list"], capture_output=True, text=True)
for line in result.stdout.split("\n"):
    if any(pkg in line.lower() for pkg in ["unsloth", "triton", "xformers", "bitsandbytes", "peft", "trl"]):
        print(line)

bitsandbytes                             0.49.2
peft                                     0.19.1
triton                                   3.6.0
trl                                      0.24.0
unsloth                                  2026.7.3
unsloth_zoo                              2026.7.3
xformers                                 0.0.35


In [6]:
from unsloth import FastVisionModel
import torch

model, processor = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-7B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
FastVisionModel.for_inference(model)
print("Model loaded in 4-bit via Unsloth.")
print(f"GPU free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

==((====))==  Unsloth 2026.7.3: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model loaded in 4-bit via Unsloth.
GPU free: 8.55 GB


In [7]:
from pathlib import Path
import datetime
import cv2

all_videos = list(Path("/kaggle/input").rglob("*.mp4"))
print(f"Found {len(all_videos)} videos:")
for v in all_videos:
    print(f"  {v}")

FRAMES_ROOT = Path("/kaggle/working/frames")
OUTPUT_ROOT = Path("/kaggle/working/qwen_outputs")
FRAMES_ROOT.mkdir(exist_ok=True)
OUTPUT_ROOT.mkdir(exist_ok=True)

video_map = {}
for v in all_videos:
    if v.name.startswith("ltx"):
        video_map["ltx"] = v
    elif v.name.startswith("hunyuan"):
        video_map["hunyuan"] = v
    elif v.name.startswith("wan"):
        video_map["wan"] = v

assert len(video_map) == 3, f"Expected 3 videos, found {len(video_map)}: {list(video_map.keys())}"

def extract_frames(video_path, output_folder, num_frames=6):
    output_folder.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    step = (total_frames - 2) / (num_frames - 1)
    frame_indices = [int(1 + i * step) for i in range(num_frames)]
    stem = video_path.stem
    for i, idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            t = idx / fps if fps > 0 else 0
            out = output_folder / f"{stem}_frame{i+1:02d}_t{t:.2f}s.jpg"
            cv2.imwrite(str(out), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
    cap.release()
    print(f"  {stem}: {num_frames} frames extracted")

for gen, video in video_map.items():
    print(f"From {video.name}:")
    extract_frames(video, FRAMES_ROOT / gen)

print("\nDone.")

Found 3 videos:
  /kaggle/input/datasets/shantanuvedanteog/msc-test-videos/wan_001_20260717_182819.mp4
  /kaggle/input/datasets/shantanuvedanteog/msc-test-videos/hunyuan_001_20260716_193318.mp4
  /kaggle/input/datasets/shantanuvedanteog/msc-test-videos/ltx_001_20260716_133303.mp4
From wan_001_20260717_182819.mp4:
  wan_001_20260717_182819: 6 frames extracted
From hunyuan_001_20260716_193318.mp4:
  hunyuan_001_20260716_193318: 6 frames extracted
From ltx_001_20260716_133303.mp4:
  ltx_001_20260716_133303: 6 frames extracted

Done.


In [9]:
from PIL import Image
import gc

ANALYSIS_PROMPT = """The 6 images below are evenly-spaced frames extracted from a single short video, in temporal order (frame 1 = earliest, frame 6 = latest). Analyse them as a sequence representing one video.

You are analysing a video for AI-generation artefacts. Please assess the following 5 categories:

1. FACIAL COHERENCE: Are faces (if any) rendered consistently across frames? Note any distortions, asymmetries, or identity drift between frames.

2. TEMPORAL CONSISTENCY: Do objects, backgrounds, and scene elements remain coherent across the frame sequence? Note any morphing, disappearing/reappearing elements, or drift.

3. PHYSICAL PLAUSIBILITY: Do movements, physics, lighting, and shadows behave naturally? Note any unnatural motion (inferred from frame-to-frame changes), incorrect shadows, or physics violations.

4. TEXTURE AND DETAIL: Are surfaces (skin, hair, fabric) rendered with realistic detail and consistency? Note any waxy skin, over-smoothed regions, or texture inconsistencies.

5. SEMANTIC COHERENCE: Does the content make logical sense? Are text, hands, and complex structures rendered correctly? Note any garbled text, extra/missing fingers, or nonsensical elements.

For each category, provide:
- A severity rating from 0 to 10, where 0 = no artefacts observed and 10 = severe artefacts
- A brief description of what you observed

Finally, provide an overall assessment: is this video likely AI-generated (yes/no/uncertain) and your confidence level (0-100%).

Structure your response in this format:

CATEGORY 1 - Facial Coherence: [rating]/10
Observations: [description]

CATEGORY 2 - Temporal Consistency: [rating]/10
Observations: [description]

CATEGORY 3 - Physical Plausibility: [rating]/10
Observations: [description]

CATEGORY 4 - Texture and Detail: [rating]/10
Observations: [description]

CATEGORY 5 - Semantic Coherence: [rating]/10
Observations: [description]

OVERALL:
Likely AI-generated: [yes/no/uncertain]
Confidence: [0-100]%
Summary: [1-2 sentence summary]"""


def analyze_video_frames(frame_dir, prompt=ANALYSIS_PROMPT):
    frame_paths = sorted(frame_dir.glob("*.jpg"))
    print(f"Found {len(frame_paths)} frames in {frame_dir.name}")
    
    # Load frames as PIL images
    images = [Image.open(str(f)).convert("RGB") for f in frame_paths]
    
    # Build message with all 6 images + prompt
    content = [{"type": "image"} for _ in images]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]
    
    input_text = processor.apply_chat_template(
        messages, add_generation_prompt=True
    )
    inputs = processor(
        images,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            use_cache=True,
        )
    
    # Trim prompt tokens from output
    trimmed = output_ids[:, inputs.input_ids.shape[1]:]
    response = processor.batch_decode(
        trimmed, skip_special_tokens=True
    )[0]
    
    # Free GPU memory between calls
    del inputs, output_ids, trimmed
    torch.cuda.empty_cache()
    gc.collect()
    
    return response


def save_response(response, generator, video_id="001"):
    header = f"""# Model: unsloth/Qwen2.5-VL-7B-Instruct (4-bit via Unsloth)
# Date: {datetime.date.today().isoformat()}
# Input: 6 frames from {generator}_{video_id}.mp4
# Prompt version: analysis_prompt_v1
# Access: Kaggle free tier (T4, Unsloth 4-bit quantization)
---
{response}
"""
    out_path = OUTPUT_ROOT / f"{generator}_{video_id}_qwen_v1.txt"
    out_path.write_text(header)
    print(f"Saved: {out_path}")


print("Prompt and functions ready.")

Prompt and functions ready.


In [19]:
print("=" * 60)
print("Analyzing LTX...")
print("=" * 60)
ltx_response = analyze_video_frames(FRAMES_ROOT / "ltx")
print("\n--- Response ---")
print(ltx_response)
save_response(ltx_response, "ltx")

Analyzing LTX...
Found 6 frames in ltx


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Response ---
CATEGORY 1 - Facial Coherence: 2/10
Observations: The woman's facial features appear slightly distorted, particularly around her eyes and mouth. There seems to be a lack of smooth transitions between frames, suggesting possible AI generation artifacts.

CATEGORY 2 - Temporal Consistency: 3/10
Observations: The background remains consistent throughout the frames, but there are minor differences in the lighting and shadows that suggest slight variations in the environment. However, these changes are not significant enough to indicate a high level of temporal inconsistency.

CATEGORY 3 - Physical Plausibility: 4/10
Observations: The woman's hair appears somewhat unnatural, with some strands appearing unnaturally straight or clumped together. The lighting on her face seems inconsistent at times, which could be due to the AI generation process.

CATEGORY 4 - Texture and Detail: 3/10
Observations: The skin texture looks somewhat artificial, lacking the natural imperfections

In [10]:
for generator in ["hunyuan", "wan"]:
    print("=" * 60)
    print(f"Analyzing {generator.upper()}...")
    print("=" * 60)
    response = analyze_video_frames(FRAMES_ROOT / generator)
    print("\n--- Response ---")
    print(response)
    save_response(response, generator)
    print()

Analyzing HUNYUAN...
Found 6 frames in hunyuan


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Response ---
CATEGORY 1 - Facial Coherence: 2/10
Observations: The woman's face appears slightly distorted in some frames, particularly around the eyes and mouth area. There seems to be a lack of smooth transitions between frames, suggesting possible AI generation artifacts.

CATEGORY 2 - Temporal Consistency: 3/10
Observations: The background remains consistent throughout the frames, but there are minor differences in the lighting and shadows that suggest slight variations in the scene's exposure or angle.

CATEGORY 3 - Physical Plausibility: 4/10
Observations: The woman's hair and facial features appear somewhat unnatural, especially when compared to real-world human characteristics. The lighting seems inconsistent, which might be due to the AI generation process.

CATEGORY 4 - Texture and Detail: 3/10
Observations: The skin texture looks somewhat artificial, lacking the natural imperfections seen on real human skin. The hair also appears overly smooth and lacks the fine details

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Response ---
CATEGORY 1 - Facial Coherence: 2/10
Observations: The woman's facial features appear slightly different in each frame, suggesting some form of distortion or identity drift. The eyes, eyebrows, and mouth positions vary subtly, which is not natural for a real person.

CATEGORY 2 - Temporal Consistency: 3/10
Observations: The background remains consistent throughout the frames, indicating that the setting is stable. However, there seems to be a slight change in the lighting intensity, which might suggest a minor inconsistency in the video's lighting setup.

CATEGORY 3 - Physical Plausibility: 4/10
Observations: The lighting and shadows appear somewhat inconsistent, especially around the hair and face. The hair strands seem to move slightly differently in each frame, which could be due to the wind or camera movement, but the transitions between frames do not seem entirely natural.

CATEGORY 4 - Texture and Detail: 5/10
Observations: The skin texture appears smooth and lac

In [11]:
import shutil
shutil.make_archive("/kaggle/working/qwen_outputs", "zip", "/kaggle/working/qwen_outputs")
!ls -la /kaggle/working/*.zip
print("\nDownload qwen_outputs.zip from the Output panel on the right sidebar.")

-rw-r--r-- 1 root root 2086 Jul 18 18:10 /kaggle/working/qwen_outputs.zip

Download qwen_outputs.zip from the Output panel on the right sidebar.
